In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [ ]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126


Looking in indexes: https://download.pytorch.org/whl/cu126


In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.7/21.7 MB 77.9 MB/s eta 0:00:00


In [ ]:
!uv pip install surya-ocr


Using Python 3.12.12 environment at: /usr
Resolved 61 packages in 587ms
Prepared 11 packages in 1.29s
Uninstalled 2 packages in 16ms
Installed 11 packages in 16ms
 + cfgv==3.5.0
 + distlib==0.4.0
 + filetype==1.2.0
 + identify==2.6.15
 + nodeenv==1.9.1
 - opencv-python-headless==4.12.0.88
 + opencv-python-headless==4.11.0.86
 - pillow==11.3.0
 + pillow==10.4.0
 + pre-commit==4.5.0
 + pypdfium2==4.30.0
 + surya-ocr==0.17.0
 + virtualenv==20.35.4


In [ ]:
!uv pip install pypdfium2

Using Python 3.12.12 environment at: /usr
Audited 1 package in 94ms


# extract word

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
import pypdfium2 as pdfium
import os


In [ ]:
# Initialize predictors once (outside the loop for efficiency)
foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()

In [ ]:
!apt-get update
!apt-get install -y libreoffice


Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,159 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,840 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,499 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-securit

In [ ]:
import os
import shutil
import subprocess
import pypdfium2 as pdfium


# Paths
input_folder = "/content/drive/MyDrive/Graduation Project/Mazen Data/Word"
output_folder = "/content/drive/MyDrive/Graduation Project/Collected Data"
temp_folder = "/content/temp_pdf"
moved_folder = "/content/drive/MyDrive/Graduation Project/Mazen Data/4"

os.makedirs(temp_folder, exist_ok=True)

# Accepted Word formats
extensions = (".docx", ".doc", ".DOC", ".rtf")

docx_files = [f for f in os.listdir(input_folder) if f.endswith(extensions)]
print(f"Found {len(docx_files)} word files to process\n")


# ---------- Convert ANY Word file → PDF using LibreOffice ----------
def convert_to_pdf(input_path):
    filename = os.path.basename(input_path)
    pdf_name = os.path.splitext(filename)[0] + ".pdf"
    output_pdf_path = os.path.join(temp_folder, pdf_name)

    command = [
        "soffice",
        "--headless",
        "--convert-to", "pdf",
        "--outdir", temp_folder,
        input_path
    ]

    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

    return output_pdf_path


# ---------- SURYA OCR ----------
def process_pdf(pdf_path, output_path):
    try:
        pdf = pdfium.PdfDocument(pdf_path)
        n_pages = len(pdf)

        print(f"Total Pages: {n_pages}")
        print("Starting OCR text extraction...\n")

        all_text = []

        for page_num in range(n_pages):
            try:
                print(f"  Processing page {page_num + 1}/{n_pages}...", end=" ")

                page = pdf[page_num]
                pil_image = page.render(scale=1, rotation=0).to_pil()

                detections = detection_predictor([pil_image])
                detected_boxes = detections[0].bboxes

                if len(detected_boxes) > 0:
                    predictions = recognition_predictor([pil_image], det_predictor=detection_predictor)
                    page_text_lines = [line.text for line in predictions[0].text_lines]
                else:
                    page_text_lines = []

                all_text.append({
                    "page": page_num + 1,
                    "full_text": "\n".join(page_text_lines)
                })

                print(f"({len(detected_boxes)} boxes, {len(page_text_lines)} lines)")

            except Exception as e:
                print(f"Error: {str(e)}")
                continue

        # Save extracted text
        with open(output_path, "w", encoding="utf-8") as f:
            for page_data in all_text:
                f.write(f"\n{'='*50}\n")
                f.write(f"PAGE {page_data['page']}\n")
                f.write(f"{'='*50}\n")
                f.write(page_data["full_text"])
                f.write("\n")

        print(f"\nOCR complete: {len(all_text)} pages processed")
        print(f"Saved: {output_path}")

    except Exception as e:
        print(f"Error processing PDF: {str(e)}")


# ---------- MAIN LOOP ----------
for idx, filename in enumerate(docx_files, 1):
    print("\n" + "="*60)
    print(f"[{idx}/{len(docx_files)}] Processing: {filename}")
    print("="*60)

    word_path = os.path.join(input_folder, filename)

    # Define output text file path
    output_filename = os.path.splitext(filename)[0] + ".txt"
    output_path = os.path.join(output_folder, output_filename)

    # 1️⃣ Skip if output already exists
    if os.path.exists(output_path):
        print("Output file already exists → Skipping extraction.")
        print("Moving file to processed folder...")
        shutil.move(word_path, moved_folder)
        continue

    print("Converting Word file → PDF (LibreOffice)...")
    pdf_path = convert_to_pdf(word_path)

    process_pdf(pdf_path, output_path)

    # 2️⃣ Move the original Word file after processing
    print("Moving processed Word file to folder 4...")
    shutil.move(word_path, moved_folder)


print("\n" + "="*60)
print("All files processed successfully!")
print("="*60)


Found 207 word files to process


[1/207] Processing: قانون المحاماة طبعة أصلية.doc
Converting Word file → PDF (LibreOffice)...
Total Pages: 93
Starting OCR text extraction...

  Processing page 1/93... 

Recognizing Text: 100%|██████████| 9/9 [00:02<00:00,  3.52it/s]


(9 boxes, 9 lines)
  Processing page 2/93... 

Recognizing Text: 100%|██████████| 1/1 [00:00<00:00,  2.55it/s]


(1 boxes, 1 lines)
  Processing page 3/93... 

...

Recognizing Text: 100%|██████████| 16/16 [00:01<00:00,  8.36it/s]


(16 boxes, 16 lines)

OCR complete: 13 pages processed
Saved: /content/drive/MyDrive/Graduation Project/Collected Data/9802I-2005-05-28_162.txt

[101/207] Processing: 9802I-2005-05-31_640-643.DOC
Converting Word file → PDF (LibreOffice)...
Total Pages: 3
Starting OCR text extraction...

  Processing page 1/3... 

Recognizing Text: 100%|██████████| 38/38 [00:03<00:00, 10.92it/s]


(38 boxes, 38 lines)
  Processing page 2/3... 

Recognizing Text:  88%|████████▊ | 37/42 [00:03<00:00, 28.18it/s]